# CIFAR-10: Data Augmentation y Arquitecturas Avanzadas

En la clase pasada entrenamos LeNet sobre FashionMNIST (imágenes en gris de 28×28) y superamos el 85% de accuracy. Hoy pasamos a **CIFAR-10**: fotos a color de 32×32 con fondos, ángulos e iluminación variables. Con este dataset, la misma LeNet alcanza aproximadamente un 55% de accuracy. En esta notebook vemos dos herramientas para mejorar ese resultado:

1. **Data augmentation**: más variedad de datos de entrenamiento sin conseguir imágenes nuevas.
2. **Una arquitectura más profunda (DenseNet)**: más capacidad, con conexiones que permiten entrenar redes de decenas de capas.

## Introducción

### Objetivos

1. **Implementar y entender técnicas de data augmentation** para aumentar la diversidad del conjunto de entrenamiento y mejorar la generalización de los modelos.
2. **Comparar arquitecturas de redes neuronales** tradicionales con arquitecturas más avanzadas como DenseNet.
3. **Entrenar y evaluar modelos de clasificación** en PyTorch utilizando el dataset CIFAR-10.
4. **Analizar el impacto de las técnicas de data augmentation y las arquitecturas avanzadas** en el rendimiento del modelo utilizando métricas adecuadas.

### Contenido

1. Configuración de bibliotecas y semillas para reproducibilidad.
2. Familiarización con `torchvision.transforms`.
3. Carga del dataset CIFAR-10 y pipelines de transformaciones.
4. Parte 1: LeNet con y sin data augmentation.
5. Parte 2: implementación de DenseNet pieza por pieza.
6. Comparación de los modelos en validación y evaluación final en test.

### Sobre el conjunto de datos

[CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) es un conjunto de datos muy utilizado en visión por computadora. Contiene 60,000 imágenes a color de 32×32 píxeles en 10 clases (avión, auto, pájaro, gato, ciervo, perro, rana, caballo, barco y camión), con 6,000 imágenes por clase: 50,000 de entrenamiento y 10,000 de test. `torchvision.datasets` lo descarga y lo carga de forma sencilla.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, Subset
from torchvision.transforms import v2 as T
from torchvision.io import read_image

from torchinfo import summary

import urllib.request
from pathlib import Path
import matplotlib.pyplot as plt

from utils import (
    train,
    plot_training,
    model_classification_report,
    show_tensor_image,
    show_tensor_images,
)

In [ ]:
# Fijamos la semilla para que los resultados sean reproducibles
SEED = 23

torch.manual_seed(SEED)
# cuDNN elige algoritmos deterministas (mismo resultado en cada corrida, algo más lento)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
from utils import get_device, get_num_workers

DEVICE = get_device()  # cuda > mps > xpu > cpu
# Linux: mitad de los núcleos disponibles, entre 1 y 8; Windows/macOS: 0 (ver docstring)
NUM_WORKERS = get_num_workers()

print(f"Usando {DEVICE}")
print(f"Usando {NUM_WORKERS} workers")

In [ ]:
# más grande que en la clase 04 (128): las imágenes son chicas y un batch grande aprovecha mejor la GPU
BATCH_SIZE = 512

## Data Augmentation

El **data augmentation** es una técnica de **regularización** que consiste en aplicar transformaciones aleatorias a las imágenes de entrenamiento: rotaciones, traslaciones, reflejos, recortes, etc. Un gato reflejado o levemente rotado sigue siendo un gato, así que la etiqueta no cambia. Como la transformación se sortea **cada vez que se lee la imagen**, la red nunca ve dos veces exactamente la misma entrada: le resulta más difícil memorizar el conjunto de entrenamiento (como pasaba con el sobreajuste de la clase 03) y tiene que aprender rasgos que resistan esos cambios.

PyTorch proporciona estas transformaciones en [`torchvision.transforms`](https://pytorch.org/vision/main/transforms.html), el mismo módulo que usamos en la clase 04 para `ToImage`, `ToDtype` y `Normalize`. Primero tomemos una imagen de ejemplo y apliquemos algunas transformaciones comunes.

In [ ]:
IMAGE_URL = "https://raw.githubusercontent.com/Catedra-IA/taller-deep-learning-letra/main/assets/dwight.jpg"
# descarga a un archivo temporal y devuelve la ruta
image_path, _ = urllib.request.urlretrieve(IMAGE_URL)

img = read_image(image_path)
show_tensor_image(img, title="Original Image")

Las transformaciones más comunes incluyen:

- [`RandomHorizontalFlip`](https://pytorch.org/vision/stable/generated/torchvision.transforms.v2.RandomHorizontalFlip.html): Voltea horizontalmente la imagen con una probabilidad dada.
- [`RandomCrop`](https://pytorch.org/vision/stable/generated/torchvision.transforms.v2.RandomCrop.html): Recorta la imagen de forma aleatoria.
- [`RandomRotation`](https://pytorch.org/vision/stable/generated/torchvision.transforms.v2.RandomRotation.html): Rota la imagen de forma aleatoria.
- [`ColorJitter`](https://pytorch.org/vision/stable/generated/torchvision.transforms.v2.ColorJitter.html): Cambia el brillo, contraste, saturación y tono de la imagen de forma aleatoria.
- [`RandomResizedCrop`](https://pytorch.org/vision/stable/generated/torchvision.transforms.v2.RandomResizedCrop.html): Recorta y cambia el tamaño de la imagen de forma aleatoria.

Se pueden encadenar con [`Compose`](https://pytorch.org/vision/stable/generated/torchvision.transforms.v2.Compose.html), igual que en la clase 04. La lista completa está en la [documentación oficial](https://pytorch.org/vision/stable/transforms.html#v2-api-reference-recommended).

In [ ]:
img_rhf = T.RandomHorizontalFlip(p=1)(img)  # p=1 para que siempre se aplique
img_rc = T.RandomCrop(size=(100, 100))(img)  # recorte de 100x100 en una posición aleatoria
img_rr = T.RandomRotation(degrees=45)(img)  # ángulo aleatorio entre -45 y 45 grados
img_cj = T.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.5)(img)
img_rrc = T.RandomResizedCrop(size=(200, 200))(img)  # recorte aleatorio llevado a 200x200

# Compose: reflejo y después recorte central
composed = T.Compose([T.RandomHorizontalFlip(p=1), T.CenterCrop(size=(100, 200))])
img_composed = composed(img)

imgs = [img, img_rhf, img_rc, img_rr, img_cj, img_rrc, img_composed]
names = ["original", "HorizontalFlip", "Crop", "Rotation", "ColorJitter", "ResizedCrop", "Compose"]
show_tensor_images(
    imgs,
    titles=[f"{n}\n({i.shape[-2]}x{i.shape[-1]})" for n, i in zip(names, imgs)],
    figsize=(18, 4),
)

Volver a ejecutar la celda anterior cambia el recorte, el ángulo y los colores: las transformaciones son **aleatorias**.

Las transformaciones aleatorias se aplican **solo a las imágenes de entrenamiento**. En validación y test queremos medir el rendimiento sobre imágenes reales, sin alterar, y que la medición sea la misma cada vez que evaluamos. Por lo tanto necesitamos dos pipelines de transformaciones: uno para entrenamiento y otro para validación/test.

> No todas las transformaciones son adecuadas para todas las tareas: la transformación no puede cambiar la etiqueta. Por ejemplo, en MNIST no se debería aplicar `RandomHorizontalFlip`, ya que los dígitos pierden su significado al voltearse, y `RandomVerticalFlip` confundiría el 6 con el 9.

## Dataset y DataLoader

Definimos una función `get_dataloaders` que devuelve los DataLoaders de entrenamiento, validación y test con las transformaciones deseadas, para poder repetir el experimento con distintos pipelines.

Hay un detalle respecto de la clase 04: `random_split` reparte **índices** sobre un único dataset base, así que train y validación compartirían el mismo `transform`, y no queremos data augmentation en validación. Por eso cargamos el conjunto de entrenamiento **dos veces**, una con cada pipeline, sorteamos una única permutación de índices y nos quedamos con el 80% de una copia (train) y el 20% restante de la otra (validación) usando [`Subset`](https://pytorch.org/docs/stable/data.html#torch.utils.data.Subset). Como los índices no se repiten, ninguna imagen queda en los dos conjuntos.

In [ ]:
DATA_DIR = Path("data")


def get_dataloaders(
    train_transf, test_transf, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
):
    """
    Función para obtener los dataloaders de entrenamiento, validación y test

    Args:
    - train_transf: transformaciones para el dataset de entrenamiento
    - test_transf: transformaciones para los datasets de validación y test
    - batch_size: tamaño del batch
    - num_workers: número de workers para cargar los datos
    """

    # descargamos el dataset CIFAR10 (si no lo tenemos ya).
    # El conjunto de entrenamiento se carga dos veces: las mismas imágenes con distinto transform
    train_base = datasets.CIFAR10(
        DATA_DIR, train=True, download=True, transform=train_transf
    )
    val_base = datasets.CIFAR10(
        DATA_DIR, train=True, download=True, transform=test_transf
    )
    test_dataset = datasets.CIFAR10(
        DATA_DIR, train=False, download=True, transform=test_transf
    )

    # dividimos los índices en entrenamiento (80%) y validación (20%)
    indices = torch.randperm(
        len(train_base),
        generator=torch.Generator().manual_seed(SEED),  # mismo split en cada llamada
    )
    train_size = int(0.8 * len(train_base))
    train_dataset = Subset(train_base, indices[:train_size].tolist())
    val_dataset = Subset(val_base, indices[train_size:].tolist())

    # creamos los dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
    )

    return train_loader, val_loader, test_loader

### Transformaciones

Vamos a definir 2 pipelines para el conjunto de entrenamiento, más el de validación/test:

1. **Sin data augmentation**: solo las transformaciones comunes de la clase 04 (`ToImage`, `ToDtype`, `Normalize`). Es el punto de referencia.
2. **Con data augmentation**: el esquema estándar para CIFAR-10, que es el que usa el paper de DenseNet: una **traslación** aleatoria de hasta 4 píxeles (`RandomCrop(32, padding=4)`: agrega 4 píxeles de borde y recorta 32×32 en una posición al azar) y un **reflejo horizontal**.

La media y el desvío de `Normalize` son un valor por canal (R, G, B), calculados sobre los píxeles de train ya escalados a $[0, 1]$.

In [ ]:
raw_dataset = datasets.CIFAR10(DATA_DIR, train=True, download=True)
name_classes = raw_dataset.classes
nclasses = len(name_classes)
print(f"Clases: {name_classes}")

# media y desvío por canal (R, G, B) de las 40,000 imágenes de train (píxeles escalados a [0, 1])
CIFAR_MEAN = [0.4911, 0.4817, 0.4459]
CIFAR_STD = [0.2471, 0.2435, 0.2615]

# Transformaciones con data augmentation
train_transforms_aug = T.Compose(
    [
        # TODO: data augmentation
        #   1. traslación: recorte aleatorio de 32x32 con 4 píxeles de padding (T.RandomCrop)
        #   2. reflejo horizontal con probabilidad 0.5 (T.RandomHorizontalFlip)
        # Transformaciones comunes:
        T.ToImage(),  # PIL -> tensor uint8 [C, H, W]
        T.ToDtype(torch.float32, scale=True),  # uint8 [0, 255] -> float32 [0, 1]
        T.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
    ]
)

# Transformaciones sin data augmentation
train_transforms_no_aug = T.Compose(
    [
        T.ToImage(),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
    ]
)

# Validación y test: nunca llevan data augmentation
test_transform = train_transforms_no_aug

Para visualizar el efecto, tomamos **una** imagen de CIFAR-10 y le aplicamos 7 veces el pipeline de data augmentation: en cada época el modelo recibe una variante distinta de la misma imagen.

In [ ]:
pil_img, label = raw_dataset[7]
augment = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(p=0.5), T.ToImage()])

show_tensor_images(
    [T.ToImage()(pil_img)] + [augment(pil_img) for _ in range(7)],
    titles=[f"original ({name_classes[label]})"] + [f"variante {i + 1}" for i in range(7)],
    figsize=(18, 3),
)

## Parte 1: Data Augmentation

En este experimento comparamos el mismo modelo entrenado con y sin data augmentation. Usamos la [LeNet](https://d2l.ai/chapter_convolutional-neural-networks/lenet.html) de la clase 04 con dos cambios: `in_channels=3` (imágenes a color) y sin `padding=2` en `c1`, porque las imágenes de CIFAR-10 ya son de 32×32, el tamaño de entrada original de LeNet.

In [ ]:
class LeNet(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(LeNet, self).__init__()
        self.c1 = nn.Conv2d(in_channels=in_channels, out_channels=6, kernel_size=5)  # 32 -> 28
        self.s2 = nn.AvgPool2d(kernel_size=2, stride=2)  # 28 -> 14
        self.c3 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)  # 14 -> 10
        self.s4 = nn.AvgPool2d(kernel_size=2, stride=2)  # 10 -> 5
        self.c5 = nn.Conv2d(in_channels=16, out_channels=120, kernel_size=5)  # 5 -> 1
        self.f6 = nn.Linear(in_features=120, out_features=84)
        self.output = nn.Linear(84, num_classes)

    def forward(self, x):
        x = F.tanh(self.c1(x))
        x = self.s2(x)
        x = F.tanh(self.c3(x))
        x = self.s4(x)
        x = F.tanh(self.c5(x))
        x = x.flatten(start_dim=1)
        x = F.tanh(self.f6(x))
        x = self.output(x)
        return x


summary(LeNet(3, nclasses), input_size=(BATCH_SIZE, 3, 32, 32))

In [ ]:
# definicion de hiperparametros
LR = 0.001
EPOCHS = 30
PATIENCE = 5
criterion = nn.CrossEntropyLoss()

### Entrenamiento sin data augmentation

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    train_transforms_no_aug, test_transform
)

lenet_no_aug = LeNet(3, nclasses).to(DEVICE)
optimizer = optim.Adam(lenet_no_aug.parameters(), lr=LR)

train_errors_ln_nda, val_errors_ln_nda = train(
    model=lenet_no_aug,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    do_early_stopping=True,
    patience=PATIENCE,
    epochs=EPOCHS,
)

Comparamos los modelos en **validación**; test lo reservamos para el modelo final (mismo criterio que en la clase 03).

In [ ]:
model_classification_report(lenet_no_aug, val_loader, DEVICE, nclasses, target_names=name_classes)

### Entrenamiento con data augmentation

In [ ]:
# TODO: repetir el experimento con data augmentation
#   1. obtener los dataloaders con train_transforms_aug (validación y test siguen con test_transform)
#   2. instanciar una LeNet nueva con el nombre lenet_aug y su optimizador Adam
#   3. entrenar con train(...) con los mismos hiperparámetros;
#      guardar las listas en train_errors_ln_da y val_errors_ln_da

In [ ]:
model_classification_report(lenet_aug, val_loader, DEVICE, nclasses, target_names=name_classes)

Superponemos las curvas de los dos entrenamientos. El aspecto relevante es la **brecha** entre la pérdida de entrenamiento (línea punteada) y la de validación (línea llena) de cada color.

In [ ]:
plt.figure(figsize=(10, 5))
for name, tr, va, color in [
    ("sin aug", train_errors_ln_nda, val_errors_ln_nda, "tab:red"),
    ("con aug", train_errors_ln_da, val_errors_ln_da, "tab:blue"),
]:
    plt.plot(tr, "--", color=color, label=f"train {name}")
    plt.plot(va, "-", color=color, label=f"val {name}")
plt.title("LeNet en CIFAR-10: con y sin data augmentation")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

**¿Qué cambió con data augmentation?**

- **Sin augmentation** la pérdida de train sigue bajando (≈ 0.98) mientras la de validación se estanca en ≈ 1.25: la red comienza a memorizar el conjunto de entrenamiento y el early stopping detiene el entrenamiento.
- **Con augmentation** la brecha desaparece. La pérdida de train queda incluso *por encima* de la de validación: las imágenes de train están trasladadas y reflejadas, son más difíciles que las de validación, que no se alteran. El early stopping ya no se activa y la pérdida de validación alcanza un valor algo menor (≈ 1.20).
- La accuracy en validación casi no cambia (56.2% → 57.6%). Data augmentation es un **regularizador**: reduce el sobreajuste, pero no le agrega capacidad al modelo. LeNet, con 62 mil parámetros, no logra ajustar adecuadamente ni siquiera el conjunto de entrenamiento (*underfitting*): la limitación está en el modelo, no en los datos.

Necesitamos una red con más capacidad, es decir, más profunda. Es el tema de la Parte 2.

## Parte 2: DenseNet

### ¿Más profundo es mejor?

LeNet tiene 5 capas con pesos y su capacidad resulta insuficiente para CIFAR-10. Una red con más capas puede, en principio, aprender características más complejas, por lo que el primer experimento es el más directo: una **LeNet mucho más profunda**. `DeepLeNet` es la misma LeNet con `extra_layers` convoluciones 3×3 adicionales (16 → 16 canales, con `padding=1` para no cambiar el tamaño) después de `c3`.

Entrenamos 5 épocas la LeNet original y versiones con 10, 20, 30 y 40 capas extra. Antes de entrenar medimos, con un batch, la norma del gradiente que le llega a la **primera** capa (`c1`).

> Sobre por qué la profundidad aporta capacidad de forma más eficiente que el ancho de las capas, se recomienda el video [*Why Deep Learning Works Unreasonably Well*](https://www.youtube.com/watch?v=qx7hirqgfuU) de Welch Labs.

In [ ]:
class DeepLeNet(nn.Module):
    def __init__(self, in_channels, num_classes, extra_layers=0):
        super(DeepLeNet, self).__init__()
        self.c1 = nn.Conv2d(in_channels, 6, kernel_size=5)
        self.s2 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.c3 = nn.Conv2d(6, 16, kernel_size=5)
        # capas extra: 16 -> 16 canales, no cambian el alto ni el ancho
        self.extra = nn.ModuleList(
            [nn.Conv2d(16, 16, kernel_size=3, padding=1) for _ in range(extra_layers)]
        )
        self.s4 = nn.AvgPool2d(kernel_size=2, stride=2)
        self.c5 = nn.Conv2d(16, 120, kernel_size=5)
        self.f6 = nn.Linear(120, 84)
        self.output = nn.Linear(84, num_classes)

    def forward(self, x):
        x = self.s2(F.tanh(self.c1(x)))
        x = F.tanh(self.c3(x))
        for conv in self.extra:
            x = F.tanh(conv(x))
        x = self.s4(x)
        x = F.tanh(self.c5(x)).flatten(start_dim=1)
        x = F.tanh(self.f6(x))
        return self.output(x)


x_batch, y_batch = next(iter(train_loader))

for extra_layers in [0, 10, 20, 30, 40]:
    deep_model = DeepLeNet(3, nclasses, extra_layers).to(DEVICE)

    # gradiente de la primera capa con un batch, antes de entrenar
    criterion(deep_model(x_batch.to(DEVICE)), y_batch.to(DEVICE)).backward()
    print(f"\n{extra_layers} capas extra | norma del gradiente de c1: {deep_model.c1.weight.grad.norm().item():.1e}")

    train_errors_deep, _ = train(
        model=deep_model,
        optimizer=optim.Adam(deep_model.parameters(), lr=LR),
        criterion=criterion,
        train_loader=train_loader,
        val_loader=val_loader,
        device=DEVICE,
        do_early_stopping=False,
        epochs=5,
    )
    plt.plot(train_errors_deep, "o-", label=f"{extra_layers} capas extra")

plt.axhline(torch.log(torch.tensor(10.0)).item(), color="gray", linestyle="--", label="azar: ln(10)")
plt.xlabel("Epochs")
plt.ylabel("Train Loss")
plt.legend()
plt.grid(True)
plt.show()

El resultado es el opuesto al esperado: cada grupo de capas adicionales empeora la pérdida, y con 40 capas extra la red **no aprende**: la pérdida permanece en $\ln(10) \approx 2.303$, el valor que corresponde a un modelo que asigna la misma probabilidad a las 10 clases. No se trata de sobreajuste, porque la que no disminuye es la pérdida de **entrenamiento**.

La causa es el **desvanecimiento del gradiente** (*vanishing gradient*). En backpropagation el gradiente de las primeras capas es un producto con un factor por cada capa posterior. Con factores menores que 1 (la derivada de `tanh` nunca supera 1) el producto tiende a 0: la norma del gradiente de `c1` disminuye entre dos y tres órdenes de magnitud por cada 10 capas adicionales, y con un gradiente de esa magnitud los pesos de las primeras capas prácticamente no se actualizan.

La solución que permitió pasar de redes de ~20 capas a redes de más de 100 es proporcionarle al gradiente un **atajo** (*skip connection*) que evite pasar por todas las capas. [ResNet](https://arxiv.org/abs/1512.03385) (2015) suma la entrada de cada bloque a su salida, $y = x + F(x)$: la *conexión residual*. [DenseNet](https://arxiv.org/abs/1608.06993) (2016) extiende la idea y **concatena** por canales, $y = [x, F(x)]$: cada capa recibe los mapas de todas las capas anteriores del bloque.

![Suma vs concatenación](https://raw.githubusercontent.com/Catedra-IA/taller-deep-learning-letra/main/assets/dense_block.png)

Consecuencias de concatenar:

- **Se reutilizan características**: los bordes que detectó la primera capa siguen disponibles, sin cambios, para la última. Cada capa puede tener pocos filtros y la red resulta eficiente en cantidad de parámetros.
- **Los canales crecen** en cada capa, y hace falta un mecanismo para reducirlos (las capas de transición).

Antes de implementar DenseNet falta presentar un componente presente en prácticamente todas las redes profundas modernas: Batch Normalization.

### Batch Normalization

[`nn.BatchNorm2d`](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html) estandariza **cada canal** con la media y la varianza del batch, y después aplica una escala y un corrimiento aprendidos:

$$ y = \gamma \cdot \frac{x - \mu_{batch}}{\sqrt{\sigma^2_{batch} + \epsilon}} + \beta $$

Es la misma idea que `Normalize` en la entrada, pero aplicada dentro de la red y en cada paso. Sus parámetros principales:

- `num_features`: cantidad de canales de la entrada. Hay un $\gamma$ (`weight`) y un $\beta$ (`bias`) por canal: son los únicos parámetros entrenables, $2 \cdot C$.
- `momentum` (0.1): en cada batch de entrenamiento actualiza un promedio móvil de la media y la varianza (`running_mean`, `running_var`). No se entrenan: son *buffers*.
- `eps`: evita dividir por cero.

En `model.train()` usa las estadísticas **del batch**; en `model.eval()` usa los promedios acumulados, para que la predicción de una imagen no dependa de qué otras imágenes hay en el batch. Por eso importa llamar a `model.train()` y `model.eval()` (la función `train` de `utils.py` ya lo hace).

In [ ]:
bn = nn.BatchNorm2d(num_features=3)
x = torch.randn(64, 3, 8, 8) * 5 + 10  # activaciones con media 10 y desvío 5


def stats(t):
    return f"media {t.mean().item():6.2f} | desvío {t.std().item():5.2f}"


print("entrada:           ", stats(x))

bn.train()  # usa la media y la varianza del batch
print("salida en train(): ", stats(bn(x)))
print("running_mean:      ", bn.running_mean)  # 0.9 * 0 + 0.1 * 10: se movió un 10% (momentum) hacia 10

bn.eval()  # usa running_mean y running_var, que todavía están lejos de las reales
print("salida en eval():  ", stats(bn(x)))
print("parámetros entrenables:", [(n, tuple(p.shape)) for n, p in bn.named_parameters()])

### Arquitectura DenseNet

Podemos dividir DenseNet en cuatro partes principales:
- **Convolución inicial**: extrae las primeras características de la imagen.
- **Bloques densos**: varias capas convolucionales conectadas entre sí por concatenación. Mantienen el alto y el ancho; aumentan los canales.
- **Transiciones**: entre bloques, reducen los canales (conv 1×1) y el alto y el ancho (pooling).
- **Capa de salida**: pooling global y una capa completamente conectada.

![Arquitectura de DenseNet (figura 2 del paper)](https://raw.githubusercontent.com/Catedra-IA/taller-deep-learning-letra/main/assets/densenet_architecture.jpeg)

#### Bloque denso

Cada capa de un [bloque denso](https://d2l.ai/chapter_convolutional-modern/densenet.html#dense-blocks) aplica, **en este orden**: `BatchNorm2d` → ReLU → convolución 3×3 con `padding=1` (no cambia alto ni ancho, condición necesaria para poder concatenar). En el `forward` concatena su entrada con su salida por la dimensión de canales con [`torch.cat`](https://pytorch.org/docs/stable/generated/torch.cat.html) (`dim=1` en un tensor `[N, C, H, W]`).

> El orden BN → ReLU → Conv se llama *pre-activación*: la entrada es una concatenación de mapas que vienen de capas distintas, con escalas distintas, y se normaliza antes de convolucionar. Como después de cada convolución viene un BatchNorm (el de la capa siguiente) que resta la media, el sesgo de la convolución es redundante: usamos `bias=False`.

El parámetro `growth_rate` ($k$ en el paper) indica cuántos canales **agrega** cada capa. Con `in_channels=6` y `growth_rate=4`, como en la figura: 6 → 10 → 14 → 18.

In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# DenseLayer                               [512, 10, 32, 32]         --
# ├─BatchNorm2d: 1-1                       [512, 6, 32, 32]          12
# ├─ReLU: 1-2                              [512, 6, 32, 32]          --
# ├─Conv2d: 1-3                            [512, 4, 32, 32]          216
# Total params: 228


class DenseLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DenseLayer, self).__init__()
        # TODO: definir bn (nn.BatchNorm2d sobre in_channels), relu y
        #       conv (3x3, padding=1, bias=False, de in_channels a out_channels)
        pass

    def forward(self, x):
        # TODO:
        #   1. out = conv(relu(bn(x)))
        #   2. devolver la concatenación de x y out por canales: torch.cat([x, out], dim=1)
        pass


dense_layer = DenseLayer(in_channels=6, out_channels=4)
summary(dense_layer, input_size=(BATCH_SIZE, 6, 32, 32))

Un `DenseBlock` encadena `num_layers` capas densas. Como cada una agrega `growth_rate` canales, la capa `i` (contando desde 0) recibe `in_channels + i * growth_rate` canales.

In [ ]:
class DenseBlock(nn.Module):
    def __init__(self, num_layers, in_channels, growth_rate):
        super(DenseBlock, self).__init__()
        # TODO: crear una lista con num_layers capas DenseLayer y guardarla en self.block como nn.Sequential(*layers).
        #       La capa i recibe in_channels + i * growth_rate canales y agrega growth_rate
        pass

    def forward(self, x):
        # TODO: aplicar self.block
        pass


# 5 capas con growth_rate=4: 6 -> 10 -> 14 -> 18 -> 22 -> 26 canales
dense_block = DenseBlock(num_layers=5, in_channels=6, growth_rate=4)
summary(dense_block, input_size=(BATCH_SIZE, 6, 32, 32), depth=2)

#### Transiciones

Dentro de un bloque denso los canales solo crecen y el alto y el ancho no cambian (si cambiaran, no se podría concatenar). Las [capas de transición](https://d2l.ai/chapter_convolutional-modern/densenet.html#transition-layers) van entre bloques y hacen las dos reducciones:

- **Canales**: una **convolución 1×1**, la que presentamos en la clase 04: combina los canales de cada píxel sin mirar vecinos. El paper llama *compresión* ($\theta$) a la fracción de canales que se conserva; con $\theta = 0.5$ se reducen a la mitad.
- **Alto y ancho**: `AvgPool2d(kernel_size=2, stride=2)`.

El orden completo es BN → ReLU → Conv 1×1 → AvgPool.

In [ ]:
class TransitionLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(TransitionLayer, self).__init__()
        # TODO: definir bn, relu, conv (1x1, bias=False) y pool (nn.AvgPool2d de 2x2 con stride 2)
        pass

    def forward(self, x):
        # TODO: bn -> relu -> conv -> pool
        pass


# 100 canales de 32x32 -> 50 canales de 16x16
transition_layer = TransitionLayer(in_channels=100, out_channels=50)
summary(transition_layer, input_size=(BATCH_SIZE, 100, 32, 32))

#### DenseNet Model

Implementamos la variante del paper para CIFAR-10: **3 bloques densos** con la misma cantidad de capas, separados por transiciones con compresión $\theta = 0.5$, y al final BN → ReLU → **pooling global** (el `AdaptiveAvgPool2d(1)` que mencionamos en la clase 04: promedia cada mapa completo y evita el `flatten` de tamaño fijo) → una capa lineal.

Para que entrene en pocos minutos reducimos la profundidad: el paper usa 12 capas por bloque o más ($L = 40$ a $190$ capas en total, entrenadas 300 épocas); nosotros usamos 6 por bloque ($L = 22$) con `growth_rate=24`, uno de los dos valores que usa el paper (el otro es 12). Tampoco incluimos el *bottleneck* 1×1 de la variante DenseNet-BC (ejercicio 5).

| Etapa | Operación | Salida `[C, H, W]` | Canales |
|---|---|---|---|
| entrada | — | `[3, 32, 32]` | |
| inicial | Conv 3×3 | `[48, 32, 32]` | $2 \cdot k$ |
| bloque 0 | 6 capas densas | `[192, 32, 32]` | $48 + 6 \cdot 24$ |
| transición 0 | Conv 1×1 + AvgPool | `[96, 16, 16]` | $192 \cdot 0.5$ |
| bloque 1 | 6 capas densas | `[240, 16, 16]` | $96 + 6 \cdot 24$ |
| transición 1 | Conv 1×1 + AvgPool | `[120, 8, 8]` | $240 \cdot 0.5$ |
| bloque 2 | 6 capas densas | `[264, 8, 8]` | $120 + 6 \cdot 24$ |
| salida | BN + ReLU + pooling global + `Linear(264, 10)` | `[10]` | un logit por clase |

In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# DenseNet                                      [512, 10]                 --
# ├─Conv2d: 1-1                                 [512, 48, 32, 32]         1,296
# ├─Sequential: 1-2                             [512, 264, 8, 8]          --
# │    └─DenseBlock: 2-1                        [512, 192, 32, 32]        141,264
# │    └─TransitionLayer: 2-2                   [512, 96, 16, 16]         18,816
# │    └─DenseBlock: 2-3                        [512, 240, 16, 16]        204,048
# │    └─TransitionLayer: 2-4                   [512, 120, 8, 8]          29,280
# │    └─DenseBlock: 2-5                        [512, 264, 8, 8]          235,440
# ├─BatchNorm2d: 1-3                            [512, 264, 8, 8]          528
# ├─ReLU: 1-4                                   [512, 264, 8, 8]          --
# ├─AdaptiveAvgPool2d: 1-5                      [512, 264, 1, 1]          --
# ├─Linear: 1-6                                 [512, 10]                 2,650
# Total params: 633,322


class DenseNet(nn.Module):
    def __init__(self, num_layers=(6, 6, 6), growth_rate=24, compression=0.5, num_classes=10):
        super(DenseNet, self).__init__()
        self.num_blocks = len(num_layers)

        # Capa inicial
        # TODO: num_channels = 2 * growth_rate; definir conv1 (3x3, padding=1, bias=False, de 3 a num_channels).
        #       No lleva BN ni ReLU: los aplica la primera capa densa (pre-activación)

        # Bloques densos y capas de transición en nn.Sequential
        # TODO: crear self.features = nn.Sequential() y, para cada bloque i:
        #   1. agregar DenseBlock(num_layers[i], num_channels, growth_rate) con self.features.add_module(nombre, modulo)
        #   2. actualizar num_channels: el bloque agrega num_layers[i] * growth_rate canales
        #   3. si no es el último bloque, agregar TransitionLayer(num_channels, int(num_channels * compression))
        #      y actualizar num_channels

        # Capa de clasificación
        # TODO: bn_final (BatchNorm2d), relu, avg_pool (nn.AdaptiveAvgPool2d(1)) y fc (Linear de num_channels a num_classes)
        pass

    def forward(self, x):
        # TODO: conv1, features, bn_final -> relu, avg_pool, torch.flatten(out, 1) y fc (sin activación: son los logits)
        pass


model = DenseNet(num_layers=(6, 6, 6), growth_rate=24, num_classes=nclasses)
summary(model, input_size=(BATCH_SIZE, 3, 32, 32), depth=2)

### Entrenamiento

Entrenamos con los dataloaders **con data augmentation** (los últimos que creamos en la Parte 1) y los mismos hiperparámetros que LeNet. El paper entrena 300 épocas con SGD y un learning rate que se reduce por etapas; nosotros usamos Adam y 30 épocas para que el entrenamiento lleve pocos minutos.

In [ ]:
densenet_model = DenseNet(num_layers=(6, 6, 6), growth_rate=24, num_classes=nclasses).to(DEVICE)
optimizer = optim.Adam(densenet_model.parameters(), lr=LR)

train_errors_dn, val_errors_dn = train(
    model=densenet_model,
    optimizer=optimizer,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    do_early_stopping=True,
    patience=10,  # la pérdida de validación de esta red es ruidosa: le damos más margen que a LeNet
    epochs=EPOCHS,
)

plot_training(train_errors_dn, val_errors_dn, mark_min=True)

Dos observaciones sobre la curva:

- **La pérdida de validación es ruidosa**, con variaciones entre épocas que LeNet no presentaba. En evaluación BatchNorm usa los promedios acumulados (`running_mean`, `running_var`), que se actualizan con retraso respecto de los pesos. En el paper el ruido desaparece cuando el learning rate se reduce; en esta notebook lo compensamos con una paciencia mayor en el early stopping.
- **Hay brecha entre train y validación** aun con data augmentation: esta red sí tiene capacidad suficiente para sobreajustar, y es en este caso donde la regularización tiene mayor efecto (ejercicio 2).

## Resultados

Primero comparamos los tres modelos en validación, con la menor pérdida que alcanzó cada uno. Después evaluamos **una sola vez** en test el modelo elegido.

In [ ]:
n_lenet = sum(p.numel() for p in lenet_aug.parameters())
n_densenet = sum(p.numel() for p in densenet_model.parameters())

print(f"{'Modelo':<22}{'Parámetros':>12}{'Mejor val loss':>16}")
print(f"{'LeNet sin aug':<22}{n_lenet:>12,}{min(val_errors_ln_nda):>16.4f}")
print(f"{'LeNet con aug':<22}{n_lenet:>12,}{min(val_errors_ln_da):>16.4f}")
print(f"{'DenseNet con aug':<22}{n_densenet:>12,}{min(val_errors_dn):>16.4f}")

In [ ]:
model_classification_report(
    densenet_model, test_loader, DEVICE, nclasses, do_confusion_matrix=True, target_names=name_classes
)

Con 10 veces más parámetros que LeNet, pero menos que una sola capa `Linear(3072, 256)`, DenseNet pasa de ≈ 57% a ≈ 84% de accuracy. En la matriz de confusión (filas: clase real; columnas: predicción) los errores se concentran en **cat ↔ dog**, animales de forma y tamaño parecidos a 32×32 píxeles, y en menor medida en **bird → dog** y **airplane → ship**. Como referencia, el paper reporta ≈ 95% en CIFAR-10 con $L = 40$, $k = 12$ y 300 épocas de entrenamiento.

## Ejercicios

1. **Atajos en `DeepLeNet`**: cambiar una línea del `forward` para que las capas extra usen una conexión residual, `x = x + F.tanh(conv(x))`, y repetir el experimento con 40 capas extra. ¿Cuánto vale ahora la norma del gradiente de `c1`? ¿La red aprende?
2. **DenseNet sin data augmentation**: entrenar la misma DenseNet con `train_transforms_no_aug` y superponer sus curvas con las del entrenamiento con augmentation (reutilizar el gráfico de la Parte 1). ¿La brecha entre train y validación cambia más o menos que en LeNet? ¿Por qué?
3. **Otras transformaciones**: agregar `RandomRotation`, `ColorJitter` u otras al pipeline y comparar en validación. ¿Todas ayudan?
4. **Tamaño de la red**: experimentar con `growth_rate` (12 como en el paper, 24) y con la cantidad de capas por bloque. ¿Qué pasa con la memoria de la GPU con `num_layers=(12, 12, 12)`, la configuración $L=40$ del paper? ¿Por qué?
5. **Bottleneck (DenseNet-B)**: agregar a `DenseLayer` una convolución 1×1 que reduzca a `4 * growth_rate` canales antes de la convolución 3×3 (BN → ReLU → Conv 1×1 → BN → ReLU → Conv 3×3). Comparar parámetros y tiempo.
6. **Dropout**: agregar `nn.Dropout` después de la convolución de cada `DenseLayer` (el paper usa 0.2 cuando no hay data augmentation) y evaluar su impacto.

#### Lectura adicional

- Welch Labs, [*Why Deep Learning Works Unreasonably Well*](https://www.youtube.com/watch?v=qx7hirqgfuU) (video): una explicación visual de por qué las redes profundas representan funciones complejas con muchas menos neuronas que una red de una sola capa oculta.
- He et al. (2015), [*Deep Residual Learning for Image Recognition*](https://arxiv.org/abs/1512.03385): ResNet y el problema de degradación.
- Huang et al. (2016), [*Densely Connected Convolutional Networks*](https://arxiv.org/abs/1608.06993): el paper de DenseNet.
- Ioffe & Szegedy (2015), [*Batch Normalization*](https://arxiv.org/abs/1502.03167).
- [Dive into Deep Learning, capítulo 8](https://d2l.ai/chapter_convolutional-modern/index.html): BatchNorm, ResNet y DenseNet.
- [Documentación de `torchvision.transforms.v2`](https://pytorch.org/vision/stable/transforms.html), con ejemplos visuales de cada transformación.